# Códigos numéricos: bifurcación logística y map tent

Este cuaderno acompaña al documento *Funciones iteradas, bifurcación logística y map tent*
(Modelos Matemáticos I) y genera sus figuras numéricas.

Cada diagrama aparece en dos versiones:

- **Código completo** (visible): el que produce las figuras del documento, con los ajustes
  de transparencia que hacen legibles a la vez las ramas delgadas y las bandas densas.
- **Versión simplificada** (secciones plegadas): el mismo procedimiento con menos puntos,
  pensado para leerse línea por línea; es el código del apéndice del documento.

Solo se usan `torch` y `matplotlib`, preinstalados en Colab; no hay nada que instalar.
Las celdas de los diagramas completos tardan alrededor de un minuto cada una.

**La idea computacional común.** Un diagrama de bifurcación pide iterar el mapa para miles
de valores del parámetro. En lugar de un ciclo `for` sobre $\mu$ y otro sobre las
iteraciones, se guarda *un estado por cada valor de* $\mu$ en un tensor y se aplica el mapa
al tensor completo: `x = mu * x * (1 - x)` actualiza las 6000 órbitas a la vez, porque las
operaciones de `torch` actúan elemento a elemento. El único ciclo que queda es el temporal
(las iteraciones), y cada figura termina siendo un `scatter` de millones de puntos.


## Diagrama de bifurcación del mapa logístico (código completo)

Para cada uno de 6000 valores de $\mu \in [2.5, 4]$ se itera $x_{n+1} = \mu x_n(1-x_n)$
desde $x_0 = 0.5$, se descartan 1000 iteraciones (el transitorio) y se grafican las 900
siguientes: lo que queda es el atractor de cada $\mu$. Los comentarios del código indican
la forma de cada tensor y la operación elemento a elemento que ejecuta cada línea.


In [ ]:
import torch
import matplotlib.pyplot as plt

# Mapa logistico: F_mu(x) = mu x (1 - x)

# Una orbita por cada valor de mu, todas iniciadas en x0 = 0.5.
mu = torch.linspace(2.5, 4.0, 6000)
# mu[j] = j-esimo valor del parametro
x  = torch.full_like(mu, 0.5)
# x[j] = estado actual de la orbita de mu[j]

n_transient = 1000
n_keep      = 900

# Transitorio: se itera sin guardar nada, solo para que cada orbita
# alcance su comportamiento asintotico.
print("Descartando el transitorio...")
for i in range(n_transient):
    x = mu * x * (1 - x)
    # x[j] = mu[j] * x[j] * (1 - x[j])
    # (las 6000 orbitas se actualizan a la vez)

# Orbita asintotica: cada iteracion se guarda como un renglon de X.
print("Recolectando la orbita asintotica...")
MU = mu.expand(n_keep, -1).clone()
# MU es de 900 x 6000, con MU[i, j] = mu[j]: la misma forma que X,
# para que cada muestra X[i, j] tenga su abscisa MU[i, j] en el scatter.
# expand repite el renglon sin copiar memoria; clone lo materializa.
X  = torch.empty(n_keep, mu.shape[0])
for i in range(n_keep):
    x = mu * x * (1 - x)
    X[i] = x
    # X[i, j] = i-esima muestra de la orbita asintotica de mu[j]

print("Graficando...")
plt.figure(figsize=(6.5, 6), dpi=200)
plt.xlim(2.5, 4.0)
plt.ylim(0, 1)
plt.xlabel(r'$\mu$', fontsize=20)
plt.ylabel(r'$x$', fontsize=20)
plt.title('Logistic Map Bifurcation Diagram', fontsize=18)
plt.tick_params(labelsize=15)

# Un color RGBA por punto: negro, con transparencia variable.
color = torch.zeros(X.shape + (4,))
# color[i, j] = (r, g, b, alfa) del punto (MU[i, j], X[i, j])

# A la izquierda el atractor consta de pocas ramas delgadas (pocos puntos
# por pixel: conviene un alfa alto); a la derecha la banda caotica llena
# el plano (miles de puntos por pixel: el mismo alfa saturaria a negro
# solido). Por eso el alfa decrece linealmente con mu, acotado por clamp.
color[:, :, 3] = torch.clamp(0.9 - 0.36 * (MU - 2.5), 0.045, 1)

plt.scatter(
    MU.flatten().numpy(),
    X.flatten().numpy(),
    color=color.reshape(-1, 4).numpy(),
    s=0.055,
    marker='.',
    linewidths=0,
)
# 900 x 6000 = 5.4 millones de puntos; el marcador es diminuto y sin
# borde porque la figura la dibuja la densidad de puntos, no su tamano.
plt.show()


## Versión simplificada: mapa logístico

El mismo procedimiento con 2000 valores del parámetro, 700 iteraciones descartadas y 300
retenidas, sin el ajuste de transparencia. Es el código del apéndice del documento.


In [ ]:
import torch
import matplotlib.pyplot as plt

# Logistic map: mu x (1 - x)

mu = torch.linspace(2.5, 4.0, 2000)
x  = torch.full_like(mu, 0.5)
X  = torch.empty(300, mu.shape[0])

for i in range(700):
    x = mu * x * (1 - x)
    # x[i] = mu[i] * x[i] * (1 - x[i])

for i in range(300):
    x = mu * x * (1 - x)
    # x[j] = mu[j] * x[j] * (1 - x[j])
    X[i] = x
    # X[i,j] = x[j]

MU = mu.expand_as(X)
# MU[i,j] = mu[j]

plt.scatter(
    MU, X,
    s=0.2, alpha=0.25,
    color='black', marker='.',
)

plt.show()


## Gráficas de la familia $T_\mu$ (código completo)

Gráficas de $T_\mu(x) = \mu\,\min(x, 1-x)$ para $\mu = 1, 1.5, 2$. Aquí no hay
iteración: solo se evalúa la función sobre una malla de $x$.


In [ ]:
import torch
import matplotlib.pyplot as plt

# Map tent: T_mu(x) = mu min(x, 1 - x)
# torch.min(x, 1 - x) elige punto a punto la rama correspondiente,
# asi que no hace falta escribir la funcion por casos.

x = torch.linspace(0, 1, 1000)

plt.figure(figsize=(7, 5))
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel(r'$x$')
plt.ylabel(r'$T_\mu(x)$')

for mu in [1.0, 1.5, 2.0]:
    T = mu * torch.min(x, 1 - x)
    plt.plot(x, T, label=fr'$\mu = {mu}$')

plt.legend()
plt.show()


## Diagrama de órbitas del map tent (código completo)

Este diagrama usa una estrategia distinta al logístico, y la diferencia es deliberada.
Para $\mu > 1$ la pendiente del map tent satisface $|T_\mu'| = \mu > 1$ en todas
partes, de modo que no hay órbitas periódicas atractoras: "descartar el transitorio y
esperar a que la órbita se estabilice" no produce nada estable. En su lugar se barre el
espacio de estados: para cada uno de 6000 valores de $\mu \in [1, 2]$ se toman 2000
condiciones iniciales y se grafica el estado de *cada una* tras 1000 iteraciones. La
figura muestra dónde termina la masa de condiciones iniciales, no una órbita.


In [ ]:
import torch
import matplotlib.pyplot as plt

# Map tent: T_mu(x) = mu min(x, 1 - x)

# Malla parametro x condicion inicial: cada columna es un valor de mu
# con 2000 condiciones iniciales equiespaciadas en [0, 1].
mu    = torch.linspace(1, 2, 6000)
x     = torch.linspace(0, 1, 2000)
X, MU = torch.meshgrid(x, mu, indexing = 'ij')
# X y MU son de 2000 x 6000:
# X[i, j]  = x[i]   (condicion inicial)
# MU[i, j] = mu[j]  (parametro)

n_iter = 1000

print("Iterando el map tent...")
for i in range(n_iter):
    X = MU * torch.min(X, 1 - X)
    # X[i, j] = mu[j] * min(X[i, j], 1 - X[i, j])
    # (12 millones de evaluaciones del mapa por iteracion)
# Al salir del ciclo, X[i, j] es el estado final de la condicion
# inicial x[i] bajo el parametro mu[j].

print("Graficando...")
plt.figure(figsize=(6.5, 6), dpi=200)
plt.xlim(1, 2)
plt.ylim(0, 1)
plt.xlabel(r'$\mu$', fontsize=20)
plt.ylabel(r'$x$', fontsize=20)
plt.title('Tent Map Bifurcation Diagram', fontsize=18)
plt.tick_params(labelsize=15)

# Un color RGBA por punto: negro, con transparencia variable.
color = torch.zeros(X.shape + (4,))
# color[i, j] = (r, g, b, alfa) del punto (MU[i, j], X[i, j])

# Para mu pequeno los estados finales se concentran en una banda angosta
# (muchos puntos por pixel: alfa bajo); al crecer mu la banda invariante
# se ensancha y los puntos se reparten, asi que el alfa crece linealmente
# con mu, acotado por clamp.
color[:, :, 3] = torch.clamp(1.5 * (MU - 1.2), 0.06, 1)

plt.scatter(
    MU.flatten().numpy(),
    X.flatten().numpy(),
    color=color.reshape(-1, 4).numpy(),
    s=0.065,
    marker='.',
    linewidths=0,
)
plt.show()


## Versión simplificada: map tent

El mismo barrido con 2000 valores del parámetro y 500 condiciones iniciales, sin el ajuste
de transparencia. Es el código del apéndice del documento.


In [ ]:
import torch
import matplotlib.pyplot as plt

# Tent-map: mu min(x, 1-x)

mu_vals = torch.linspace(1, 2, 2000)
x_vals  = torch.linspace(0, 1, 500)
x, mu = torch.meshgrid(
    x_vals, mu_vals, indexing='ij')
# x[i,j]  = x_vals[i]
# mu[i,j] = mu_vals[j]

for i in range(1000):
    x = mu * torch.min(x, 1 - x)
    # torch.min(x, 1 - x)[i,j] =
    #   min(x[i,j], 1 - x[i,j])
    # x[i,j] = mu[i,j] *
    #   min(x[i,j], 1 - x[i,j])

plt.scatter(
    mu, x,
    s=0.25, alpha=0.4,
    color='black', marker='.',
)
plt.show()
